<a href="https://colab.research.google.com/github/duyb2207513/Hugging_Face_Models_Mastery_RandD_Guideline/blob/main/Step6_LoRA_finetune_11_5_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 6: Deep Dive into LoRA**

#**Research/Learn**:
LoRA theory in depth (low-rank adaptation, matrix
decomposition, parameter efficiency, target modules, mathematical foundation).

#**Low-rank adaptation (LoRA)**  
LoRA is a technique used to adapt machine learning models to new contexts. It can adapt large models to specific uses by adding lightweight pieces to the original model rather than changing the entire model. A data scientist can quickly expand the ways that a model can be used rather than requiring them to build an entirely new model.
#Matrix Decomposition
LoRA apply matrix decomposition technique. By that way, LoRA is built on the understanding that large models inherently possess a low-dimensional structure. By leveraging smaller matrices, which are called low-rank matrices, LoRA adapts these models effectively. This method focuses on the core concept that significant model changes can be represented with fewer parameters, thus making the adaptation process more efficient.

**#Mathematical Foundation**
A layer have formula: y=Wx(W:weighed matrix, x :input batch, y:label). And tradditional fine tune model add delta_W to original Weighed, it's spend alot of time and GPU. And with LoRA technique have formula: y=Wx+ BAx. it  freeze originial.

**#Parameter Efficiency**

One of the biggest advantages of LoRA is parameter efficiency. Traditional fine-tuning updates all parameters of the model, which requires huge GPU memory and training cost. In contrast, LoRA only trains the low-rank matrices A and B while freezing the original pretrained weights.

Instead of training a full matrix: A*B,LoRA only trains: A*r + r*B

where:
* r is a very small rank
* r≪A,B

This significantly reduces the number of trainable parameters and memory usage.
For example, a 4096*4096 matrix contains more than 16 million parameters, while LoRA with rank r=8 only trains around 65 thousand parameters.

**#Target Modules**

LoRA is usually applied to specific modules inside Transformer architectures instead of the entire model. These modules are called target modules.

Common target modules include:

Query projection (Wq​)
Key projection (Wk)
Value projection (Wv)
Output projection (Wo​)

These layers are important because they directly affect the attention mechanism of the Transformer.


In [ ]:
!pip install peft trl torchao>=0.16.0 --upgrade

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# 2. DATASET

dataset = load_dataset("fawern/Text-to-sql-query-generation")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/277 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
print(dataset['train'][0])

{'prompt': ' <s> [INST] You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.\n      Question : Name the home team for carlton away team [/INST] SQL Query : SELECT home_team FROM table_name_77 WHERE away_team = "carlton" </s>'}


In [ ]:

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # Changed from torch_dtype to dtype
    device_map="auto"
)


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
import re

def format_chat(example):

    text = example["prompt"]

    # lấy phần [INST] ... [/INST]
    inst_text = re.search(
        r"\[INST\](.*?)\[/INST\]",
        text,
        re.S
    ).group(1).strip()

    # tách system + question
    parts = inst_text.split("Question :")

    system_message = parts[0].strip()
    question = parts[1].strip()

    # lấy SQL query
    query = re.search(
        r"SQL Query :(.*?)</s>",
        text,
        re.S
    ).group(1).strip()

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": question
        },
        {
            "role": "assistant",
            "content": query
        }
    ]

    return {"messages": messages}


In [ ]:
chat= format_chat(dataset['train'][0])
print(chat)

{'messages': [{'role': 'system', 'content': 'You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.'}, {'role': 'user', 'content': 'Name the home team for carlton away team'}, {'role': 'assistant', 'content': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}]}


In [ ]:
formated_dataset = dataset.map(format_chat)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
def formatting(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}
Qwen_template_dataset=formated_dataset.map(formatting)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:

print(Qwen_template_dataset['train'][0])

{'prompt': ' <s> [INST] You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.\n      Question : Name the home team for carlton away team [/INST] SQL Query : SELECT home_team FROM table_name_77 WHERE away_team = "carlton" </s>', 'messages': [{'role': 'system', 'content': 'You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.'}, {'role': 'user', 'content': 'Name the home team for carlton away team'}, {'role': 'assistant', 'content': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}], 'text': '<|im_start|>system\nYou are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.<|im_end|>\n<|im_start|>user\nName the home team for carlton away team<|im_end|>\n<|im_start|>assistant\nSELECT home_team FROM table_name_77 WHERE away_team = "carlton"<|im_end|>\n'}


In [ ]:
Qwen_template_dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'messages', 'text'],
        num_rows: 10000
    })
})

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
    )

tokenized_ds = Qwen_template_dataset.map(tokenize)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
tokenized_ds = tokenized_ds.map(

    remove_columns=[col for col in tokenized_ds["train"].column_names if col not in ["input_ids", "attention_mask"]]
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
token= tokenized_ds['train'][0]
token

{'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  7870,
  3239,
  13823,
  320,
  1318,
  4686,
  1331,
  1470,
  568,
  4615,
  3383,
  374,
  311,
  6923,
  264,
  7870,
  3239,
  504,
  279,
  2661,
  3405,
  13,
  151645,
  198,
  151644,
  872,
  198,
  675,
  279,
  2114,
  2083,
  369,
  1803,
  75,
  777,
  3123,
  2083,
  151645,
  198,
  151644,
  77091,
  198,
  4858,
  2114,
  26532,
  4295,
  1965,
  1269,
  62,
  22,
  22,
  5288,
  3123,
  26532,
  284,
  330,
  6918,
  75,
  777,
  1,
  151645,
  198],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [ ]:
print(tokenizer.decode(token['input_ids']))

<|im_start|>system
You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.<|im_end|>
<|im_start|>user
Name the home team for carlton away team<|im_end|>
<|im_start|>assistant
SELECT home_team FROM table_name_77 WHERE away_team = "carlton"<|im_end|>



In [ ]:
dataset = tokenized_ds["train"].train_test_split(
    test_size=0.1,
    seed=42
)

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [ ]:
dataset['train'][0]

{'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  7870,
  3239,
  13823,
  320,
  1318,
  4686,
  1331,
  1470,
  568,
  4615,
  3383,
  374,
  311,
  6923,
  264,
  7870,
  3239,
  504,
  279,
  2661,
  3405,
  13,
  151645,
  198,
  151644,
  872,
  198,
  3838,
  374,
  279,
  220,
  17,
  15,
  16,
  16,
  13369,
  5264,
  323,
  264,
  220,
  17,
  15,
  16,
  15,
  1207,
  37,
  30,
  151645,
  198,
  151644,
  77091,
  198,
  4858,
  330,
  17,
  15,
  16,
  16,
  1,
  4295,
  1965,
  62,
  16,
  17,
  22,
  23,
  22,
  5288,
  330,
  51,
  9783,
  1,
  284,
  364,
  2863,
  495,
  10480,
  1787,
  6,
  3567,
  330,
  17,
  15,
  16,
  15,
  1,
  284,
  364,
  80,
  69,
  6,
  151645,
  198],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(model_id)

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     torch_dtype="auto",
#     device_map="auto"
# )

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=10,
    save_steps=100,

    eval_strategy="steps",
    eval_steps=100,

    fp16=True,

    report_to="none"
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

#For following training and for knowing how many epoch is enough, so i train step-by-step

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)

trainer.train()

Step,Training Loss,Validation Loss
100,1.056190,0.982183
200,0.900692,0.902797
300,0.861330,0.865221
400,0.861325,0.843747
500,0.863093,0.832426


TrainOutput(global_step=563, training_loss=0.9430600749874624, metrics={'train_runtime': 957.322, 'train_samples_per_second': 9.401, 'train_steps_per_second': 0.588, 'total_flos': 2356988241907200.0, 'train_loss': 0.9430600749874624, 'epoch': 1.0})

After 1 epochs, avg_loss:0.94, total_training: 957s. Continue training with epoch 2

In [ ]:
training_args.num_train_epochs = 2
trainer.train(resume_from_checkpoint=True)

Step,Training Loss,Validation Loss
600,0.830365,0.829675
700,0.808219,0.815819
800,0.825675,0.808260
900,0.771102,0.799715
1000,0.764262,0.794874
1100,0.799900,0.790915


TrainOutput(global_step=1126, training_loss=0.3957359786567214, metrics={'train_runtime': 989.8437, 'train_samples_per_second': 18.185, 'train_steps_per_second': 1.138, 'total_flos': 4713807975360000.0, 'train_loss': 0.3957359786567214, 'epoch': 2.0})

global_step=1126, avg_loss: 0.79

In [ ]:
training_args.num_train_epochs = 3
trainer.train(resume_from_checkpoint=True)

Step,Training Loss,Validation Loss
1200,0.737871,0.792122
1300,0.764958,0.786303
1400,0.696266,0.780807
1500,0.714196,0.778945
1600,0.755639,0.775574


TrainOutput(global_step=1689, training_loss=0.2510092293743271, metrics={'train_runtime': 919.4161, 'train_samples_per_second': 29.366, 'train_steps_per_second': 1.837, 'total_flos': 7069461111820800.0, 'train_loss': 0.2510092293743271, 'epoch': 3.0})

In [ ]:
# Xem lịch sử loss đầy đủ
import pandas as pd
log_history = trainer.state.log_history
df = pd.DataFrame(log_history)
print(df[['step','loss','eval_loss']].dropna(subset=['loss']))

     step      loss  eval_loss
0      10  2.366581        NaN
1      20  1.341816        NaN
2      30  1.207614        NaN
3      40  1.140209        NaN
4      50  1.089013        NaN
..    ...       ...        ...
179  1640  0.736939        NaN
180  1650  0.746081        NaN
181  1660  0.745760        NaN
182  1670  0.760319        NaN
183  1680  0.701884        NaN

[168 rows x 3 columns]


In [ ]:
training_args.num_train_epochs = 5
trainer.train(resume_from_checkpoint=True)

Step,Training Loss,Validation Loss
1700,0.717787,0.778574
1800,0.720574,0.776415
1900,0.716282,0.773050
2000,0.687564,0.767914
2100,0.772989,0.762560
2200,0.710761,0.760370
2300,0.669134,0.758524
2400,0.669358,0.756273
2500,0.724296,0.754560
2600,0.667518,0.752630


TrainOutput(global_step=2815, training_loss=0.2865312905125254, metrics={'train_runtime': 1776.8474, 'train_samples_per_second': 25.326, 'train_steps_per_second': 1.584, 'total_flos': 1.17785983784832e+16, 'train_loss': 0.2865312905125254, 'epoch': 5.0})

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ===== Base model =====
base_model = "Qwen/Qwen2.5-0.5B-Instruct"

# ===== LoRA adapter path =====
lora_path = "./qwen_sql_lora/checkpoint-2815"

# ===== Load tokenizer =====
tokenizer = AutoTokenizer.from_pretrained(base_model)

# ===== Load base model =====
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ===== Load LoRA adapter =====
model = PeftModel.from_pretrained(
    model,
    lora_path
)
# ===== Inference mode =====
model.eval()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Show all customers from Vietnam
assistant
SELECT DISTINCT customer_id FROM customers WHERE location = 'Vietnam'


In [ ]:


# ===== Test question =====
messages = [
    {
        "role": "system",
        "content": " You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question."
    },
    {
        "role": "user",
        "content": "Show all customers from Vietnam"
    }
]


# ===== Apply Qwen chat template =====
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# ===== Tokenize =====
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ===== Generate =====
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=False
    )

# ===== Decode =====
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Show all customers from Vietnam
assistant
SELECT DISTINCT customer_id FROM customers WHERE location = 'Vietnam'


In [ ]:
message_2 = [
    {
        "role": "system",
        "content": " You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question."
    },
    {
        "role": "user",
        "content": "Users with highest reputation both in SO and Math ( geometric mean = average digits)"
    }
]

In [ ]:

# ===== Apply Qwen chat template =====
text = tokenizer.apply_chat_template(
    message_2,
    tokenize=False,
    add_generation_prompt=True
)

# ===== Tokenize =====
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ===== Generate =====
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=False
    )

# ===== Decode =====
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Users with highest reputation both in SO and Math ( geometric mean = average digits)
assistant
SELECT DISTINCT LOWER(LOWER(username)) AS username, LOWER(LOWER(email)) AS email FROM users WHERE (1 + RAND() * 200) > (SELECT MAX(CASE WHEN t1.reputation >= 5 THEN Reputation ELSE 0 END) FROM (SELECT COUNT(*) AS "reputation", ROW_NUMBER() OVER (ORDER BY Reputation DESC) AS "rank" FROM users AS t1 JOIN user_reputation AS t2 ON t1.id = t2.user_id GROUP BY t1.id) AS q1) AS p1 JOIN (SELECT COUNT(*) AS "reputation", ROW_NUMBER() OVER (ORDER BY Reputation


In [ ]:
from huggingface_hub import login

login()

In [ ]:
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_16"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql-lora_rank_16"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp__fqpof8/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  39%|###8      | 3.36MB / 8.68MB            

CommitInfo(commit_url='https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_16/commit/6ff5b9a19300b1138b3ef1d6caf77658c6949b05', commit_message='Upload model', commit_description='', oid='6ff5b9a19300b1138b3ef1d6caf77658c6949b05', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/qwen-text2sql-lora_rank_16', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/qwen-text2sql-lora_rank_16'), pr_revision=None, pr_num=None)

original_message_2=

""" You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
Question : Users with highest reputation both in SO and Math ( geometric mean = average digits). SQL Query : SELECT s.DisplayName, s.Reputation AS RepSO, m.Reputation AS RepMath, (LOG10(s.Reputation) + LOG10(m.Reputation)) / 2 AS RepAvDigits FROM "stackexchange.math".Users AS m, "stackoverflow".Users AS s WHERE s.Reputation > 10000 AND m.Reputation > 10000 AND s.AccountId = m.AccountId ORDER BY 4 DESC </s>
"""
#REVIEWS:
With complex or hard SQL queries, the model tends to hallucinate and generate incorrect tables, columns, or query structures. This happens because the model only has 0.5B parameters, which limits its reasoning and schema understanding capabilities. While the model performs well on simple SQL generation tasks, it struggles with multi-table joins, nested queries, aggregation logic, and cross-domain reasoning. Smaller language models mainly learn token patterns rather than deep semantic understanding, so they are more likely to hallucinate when the query becomes complicated.
